In [2]:
# Código para resolver o problema de tomografia quântica para o caso de 1 q-bit.
import numpy as np
from scipy.optimize import minimize

In [ ]:
# Criando as matrizes de Pauli e a matriz identidade
I = np.eye(2, dtype=complex)

# X
sigma_x = np.array([[0, 1], 
                    [1, 0]], dtype=complex)
# Y
sigma_y = np.array([[0, -1j], 
                    [1j, 0]], dtype=complex)
# Z
sigma_z = np.array([[1, 0], 
                    [0, -1]], dtype=complex)


In [ ]:
# O objetivo dessa classe é apenas gerar o estado "desconhecido"
class QuantState:
    def __init__(self, estado):
        """
        Aceita um vetor 1D (estado puro) ou uma matriz 2D (matriz densidade / estado misto).
        """
        estado = np.array(estado, dtype=complex)
        
        if estado.ndim == 1:
            # Estado Puro: Vetor de estado |psi>
            # Normaliza o vetor por segurança
            norma = np.linalg.norm(estado)
            psi = estado / norma
            
            # Converte |psi> para a matriz densidade rho = |psi><psi|
            self.rho = np.outer(psi, psi.conj())
            
        elif estado.ndim == 2:
            # Estado Misto (ou matriz densidade já pronta)
            # Verifica se é uma matriz quadrada
            if estado.shape[0] != estado.shape[1]:
                raise ValueError("A matriz densidade deve ser quadrada.")
            
            # Normaliza o traço para garantir Tr(rho) = 1
            traco = np.trace(estado)
            self.rho = estado / traco
        else:
            raise ValueError("O estado deve ser um vetor 1D (|psi>) ou matriz 2D (rho).")
    
    def measure(self, obs):
        """
        Mede o estado em relação a um observável (matriz hermitiana).
        Retorna o valor esperado <obs> = Tr(rho * obs).
        """
        return np.trace(self.rho @ obs).real  # Retorna apenas a parte real   

In [4]:
def parametros_para_T(params):
    """
    Recebe um vetor 1D de 4 numeros reais [t0, t1, t2, t3]
    e constroi a matriz triangular complexa T (2x2).
    """
    t0, t1, t2, t3 = params
    
    # Monta a matriz triangular inferior com numeros complexos
    T = np.array([
        [t0, 0],
        [complex(t2, t3), t1]
    ], dtype=complex)
    
    return T


def parametros_para_rho(params):
    """
    Converte os parametros reais diretamente na matriz densidade rho valida.
    Fórmula: rho = (T_dag * T) / Tr(T_dag * T)
    """
    T = parametros_para_T(params)
    
    # Multiplicação T^\dagger @ T
    T_dagger = T.conj().T
    T_dag_T = T_dagger @ T
    
    # Normalização pelo Traço
    traco = np.trace(T_dag_T)
    
    # Evita divisão por zero se T for totalmente nula
    if np.isclose(traco, 0):
        return np.eye(2, dtype=complex) / 2.0
        
    rho = T_dag_T / traco
    return rho

In [ ]:


def funcao_de_custo(params, mediadores, dados_exp):
    """
    params: Vetor de parametros reais ajustados pelo otimizador [t0, t1, t2, t3]
    mediadores: Lista de POVMs (Q_i) ou Observaveis (A_k)
    dados_exp: Vetor com os dados medidos no lab (probabilidades q_i ou valores esperados <A_k>)
    """
    # 1. Gera o rho candidato para os parametros atuais
    rho_candidato = parametros_para_rho(params)
    
    # 2. Calcula as previsoes teoricas para cada medidor
    previsoes = [np.real(np.trace(rho_candidato @ Q)) for Q in mediadores]
    previsoes = np.array(previsoes)
    
    # 3. Calcula o Erro Quadratico Medio
    erro = np.sum((previsoes - dados_exp) ** 2)
    return erro

In [ ]:
# Teste simples
unk_state = QuantState([1, 0])  # Estado |0>
mediadores_Q = [sigma_x, sigma_z]  # Lista com seus POVMs ou Observáveis
b_medidos = [unk_state.measure(sigma_x), unk_state.measure(sigma_z)]    # Seus dados experimentais
chute_inicial = np.random.randn(4)

resultado = minimize(
    funcao_de_custo, 
    x0=chute_inicial, 
    args=(mediadores_Q, b_medidos),
    method='Nelder-Mead'  # Excelente para otimização sem derivadas explícitas
)